# Module detection (Python) - Get pseudobulk objects for EACH cell type

In [ ]:
import pseudobulk_analytics_utils_04 as pau
import scanpy as sc
import anndata as ad
import decoupler as dc
import gseapy as gp
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

import glob
import os
from openpyxl.utils import get_column_letter
import seaborn as sns


%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import scProportionTest as pt

## 1. Load in file and keep transcripts of interest

### A. Load correct file

In [ ]:
adata_query_X =ad.read_h5ad("/tscc/lustre/ddn/scratch/aopatel/adata_MTG_Finalized.h5ad")

In [ ]:
#### Make sure UMAP is same one as in 3b

sc.pl.umap(
    adata_query_X,
    color="predictions",
    frameon=False,
    legend_loc="on data",   # puts numbers on clusters
    legend_fontsize=10,
    legend_fontoutline=2
)

### ------Only Applicable to SEA-AD (Start) --------

#### A1. Incorporate and filter based on MMSE

In [ ]:
# ── Load individual metadata ───────────────────────────────────────────────────
indiv_meta = pd.read_csv('/tscc/nfs/home/aopatel/synapse_meta_NEW/SEA-AD_individual_metadata.csv')

# Check what's in it
print(indiv_meta.shape)
print(indiv_meta.columns.tolist())
print(indiv_meta.head(3))

In [ ]:
# ── Build lookup dictionaries ──────────────────────────────────────────────────
mmse_map = indiv_meta.set_index('individualID')['MMSE score']
casi_map  = indiv_meta.set_index('individualID')['CASI score']

# ── Map onto obs using individualID ───────────────────────────────────────────
adata_query_X.obs['MMSE score'] = adata_query_X.obs['individualID'].map(mmse_map)
adata_query_X.obs['CASI score'] = adata_query_X.obs['individualID'].map(casi_map)

# ── Convert to numeric ─────────────────────────────────────────────────────────
adata_query_X.obs['MMSE score'] = pd.to_numeric(adata_query_X.obs['MMSE score'], errors='coerce')
adata_query_X.obs['CASI score'] = pd.to_numeric(adata_query_X.obs['CASI score'], errors='coerce')

# ── Verify ─────────────────────────────────────────────────────────────────────
check = (adata_query_X.obs
    .drop_duplicates('individualID')
    [['individualID', 'comparison_group', 'MMSE score', 'CASI score']]
    .sort_values('comparison_group')
)
print(check.to_string())
print(f"\nMMSE missing donors: {check['MMSE score'].isna().sum()}")
print(f"CASI missing donors: {check['CASI score'].isna().sum()}")

In [ ]:
#### MMSE FILTER

adata_query_X = adata_query_X[
    (
        (adata_query_X.obs['comparison_group'] == 'AD')
    ) |
    (
        (adata_query_X.obs['comparison_group'] == 'PathologyControl') &
        (adata_query_X.obs['MMSE score'] > 23)
    )
].copy()

print(f"MMSE-filtered: {adata_query_X.n_obs} cells, {adata_query_X.obs['individualID'].nunique()} donors")
print(adata_query_X.obs.drop_duplicates('individualID')['comparison_group'].value_counts())

In [ ]:
##### AD DONOR SEPARATION BASED ON MMSE-linked CLUSTERS

# ── Define donor lists ─────────────────────────────────────────────────────────
high_mmse_donors = [
    'H20.33.004', 'H20.33.026', 'H20.33.029', 'H20.33.031', 'H21.33.005',
    'H21.33.007', 'H21.33.027', 'H21.33.029', 'H21.33.042'
]

low_mmse_donors = [
    'H20.33.020', 'H20.33.028', 'H20.33.037', 'H21.33.008', 'H21.33.009',
    'H21.33.039', 'H21.33.044', 'H21.33.046'
]

# ── Build comparison_group_2 ──────────────────────────────────────────────────
def assign_group2(row):
    if row['comparison_group'] == 'PathologyControl':
        return 'PathologyControl'
    elif row['individualID'] in high_mmse_donors:
        return 'high_MMSE'
    elif row['individualID'] in low_mmse_donors:
        return 'low_MMSE'
    else:
        return 'not_annotated'

adata_query_X.obs['comparison_group_2'] = adata_query_X.obs.apply(assign_group2, axis=1)

# ── Verify ─────────────────────────────────────────────────────────────────────
print(adata_query_X.obs['comparison_group_2'].value_counts())
print()
print(adata_query_X.obs.drop_duplicates('individualID')[
    ['individualID', 'comparison_group', 'comparison_group_2', 'MMSE score']
].sort_values('comparison_group_2').to_string())

In [ ]:
print(adata_query_X.obs['comparison_group_2'].unique())

In [ ]:
print(pd.crosstab(
    adata_query_X.obs.drop_duplicates('individualID')['Braak'],
    adata_query_X.obs.drop_duplicates('individualID')['comparison_group_2']
))

### ------Only Applicable to SEA-AD (End) --------

### Start Pseudobulking the same way as 04

### B. Filter for important genes and keep high population cell types

In [ ]:
#### Get rid of all mitochondrial and unannotated gene transcripts

## Create mask
mask = ~(adata_query_X.var_names.str.startswith('MT-') | 
         adata_query_X.var_names.str.startswith('ENSG'))

## Filter
adata_query_X = adata_query_X[:, mask].copy()
print(f"Remaining genes: {adata_query_X.n_vars}")

In [ ]:
#### Get rid of cells types that have <3000 cells total

adata_query_X=pau.low_cell_type_remover(adata_query_X,n=3000)

In [ ]:
### Find diagnostic breakdown Severely Affected Donors

donor_breakdown = (adata_query_X.obs
    .drop_duplicates(subset='individualID')
    [['individualID', 'Severely Affected Donor', 'comparison_group']]
    .groupby(['Severely Affected Donor', 'comparison_group'])
    .size()
    .unstack(fill_value=0)
)
print(donor_breakdown)

## 2. Use "decoupler" for pseudobulking

<div class="alert alert-block alert-info">
<b> Use decoupler to pseudobulk the snRNAseq data. Sum counts by donor. Make sure to use raw counts
</div>

In [ ]:
pdata = dc.pp.pseudobulk(
    adata_query_X, 
    sample_col='individualID', 
    groups_col='predictions', 
    layer='counts',    # Targets your raw data
    mode='sum'        # Sums the counts
)

print(pdata)

<div class="alert alert-block alert-info">
<b> pseudobulking changes meta data types, these must be changed to correct type again!! 
</div>

In [ ]:
pdata.obs['ageDeath_numeric'] = pd.to_numeric(pdata.obs['ageDeath_numeric'], errors='coerce')
pdata.obs['pmi'] = pd.to_numeric(pdata.obs['pmi'], errors='coerce')
pdata.obs['RIN'] = pd.to_numeric(pdata.obs['RIN'], errors='coerce')
pdata.obs['CPS'] = pd.to_numeric(pdata.obs['CPS'], errors='coerce')
pdata.obs['sex'] = pdata.obs['sex'].astype('category')
pdata.obs['comparison_group_2'] = pdata.obs['comparison_group_2'].astype('category')

# Verify
print(pdata.obs[['sex', 'ageDeath_numeric', 'pmi', 'comparison_group_2', 'RIN', 'CPS']].dtypes)

In [ ]:
pdata_filtered = {}

for pred in pdata.obs['predictions'].unique():  
    adata_sub = pdata[pdata.obs['predictions'] == pred].copy() 
    
    # Perform filter 
    dc.pp.filter_by_expr(
        adata=adata_sub,
        group="comparison_group_2",
        min_count=10,
        min_total_count=15,
        large_n=10,
        min_prop=0.7,
    )
    dc.pp.filter_by_prop(
        adata=adata_sub,
        min_prop=0.1,
        min_smpls=2,
    )
    
    pdata_filtered[pred] = adata_sub

for pred, adata_sub in pdata_filtered.items():
    print(f"{pred}: {adata_sub.n_obs} donors, {adata_sub.n_vars} genes")

## 3. Save cell type specific RAW counts matrices for each cell type for (CEMiTool in R)

In [ ]:
OUTDIR = "MTG_cemitool_input"
os.makedirs(OUTDIR, exist_ok=True)

for pred, adata_sub in pdata_filtered.items():
    # filename-safe cell type (same convention as your DESeq2 outputs)
    ct = pred.replace('/', '-').replace(' ', '_')

    # ── expression: genes as ROWS, samples as COLUMNS ─────────────────────────
    X = adata_sub.X.toarray() if hasattr(adata_sub.X, "toarray") else adata_sub.X
    expr = pd.DataFrame(
        np.asarray(X).T.astype(int),      # transpose → genes × samples
        index=adata_sub.var_names,        # genes
        columns=adata_sub.obs_names       # samples (donors)
    )
    expr.index.name = "gene"
    expr.to_csv(f"{OUTDIR}/{ct}_counts.tsv", sep="\t")

    # ── sample annotation: CEMiTool defaults are SampleName / Class ───────────
    annot = pd.DataFrame({
        "SampleName": adata_sub.obs_names,
        "Class": adata_sub.obs["comparison_group_2"].astype(str).values,
    })
    annot.to_csv(f"{OUTDIR}/{ct}_annot.tsv", sep="\t", index=False)

    print(f"{pred:16} → {expr.shape[0]} genes × {expr.shape[1]} samples | "
          f"{annot['Class'].value_counts().to_dict()}")

## 4. Follow 05b (in different env) and return after finished

## 5. Continue with module differential enrichment

### Choose cell type of interest

In [ ]:
CT = "L4_IT"   # filename-safe cell type

# ── ranking: your existing DESeq2 stat column ────────────────────────────────
deg = pd.read_csv(f"MTG_MMSE_pb_DEGs_low/{CT}_deseq2_results.csv", index_col=0)
rnk = (deg['stat'].dropna()
       .sort_values(ascending=False)
       .rename_axis('gene').reset_index())      # two cols: gene, stat

# ── run prerank against the CEMiTool modules ─────────────────────────────────
pre = gp.prerank(
    rnk=rnk,
    gene_sets=f"MTG_cemitool_input/{CT}_cemitool_modules.gmt",
    min_size=5,
    max_size=2000,          # modules are large — raise this or big ones get dropped
    permutation_num=1000,
    seed=42,
    no_plot=True,
    outdir=None,
)

res = pre.res2d.copy()
res['NES']   = res['NES'].astype(float)
res['FDR q-val'] = res['FDR q-val'].astype(float)
res = res.sort_values('NES', key=abs, ascending=False)
print(res[['Term','ES','NES','NOM p-val','FDR q-val','Lead_genes']].to_string())

In [ ]:
import os, glob
print(glob.glob("MTG_cemitool_input/*modules*.gmt"))

### RUN GSEA using CELL TYPE SPECIFIC MODULE as library for enrichment

In [ ]:
import gseapy as gp

def load_gmt(path):
    mods = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split('\t')
            mods[parts[0]] = parts[2:]
    return mods

mods_l4 = load_gmt("MTG_cemitool_input/L4_IT_cemitool_modules.gmt")
print({m: len(g) for m, g in mods_l4.items()})

for m in sorted(mods_l4.keys()):
    try:
        enr = gp.enrichr(gene_list=mods_l4[m],
                         gene_sets=['GO_Biological_Process_2021'],
                         outdir=None)
        top = enr.results.sort_values('Adjusted P-value')[['Term','Adjusted P-value']].head(6)
        print(f"\n=== L4 IT {m}  ({len(mods_l4[m])} genes) ===")
        print(top.to_string(index=False))
    except Exception as e:
        print(f"\n=== L4 IT {m} — enrichment failed: {e} ===")

### Sample plot, change according to you above results (MMSE_p/MMSE_rho found in 05b with eigen gene correlation analysis)

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns

l4 = pd.DataFrame({
    'module':   ['M4','M3','M1','M2','M5'],
    'GSEA_NES': [ 3.09, -1.75, -2.05, -1.94, -1.61],
    'GSEA_FDR': [ 0.000, 0.000, 0.000, 0.000, 0.0004],
    'MMSE_rho': [-0.29,  0.31,  0.30, -0.06,  0.30],
    'MMSE_adjp':   [ 0.071, 0.051, 0.056, 0.689, 0.059],
    'label':    ['Axon guidance/semaphorin','Unfolded-protein response',
                 'Cell-cell adhesion','Ribosomal/translation','Unnamed (NS)'],
}).set_index('module')

rows = [f"{m}: {l4.loc[m,'label']}" for m in l4.index]

# numeric matrix — explicitly float
mat = pd.DataFrame(
    {'GSEA NES\n(up in low_MMSE)': l4['GSEA_NES'].astype(float).values,
     'MMSE corr\n(−rho)':          (-l4['MMSE_rho'].astype(float)).values},
    index=rows,
)

# string annotations kept SEPARATE
ann = pd.DataFrame(
    {'GSEA NES\n(up in low_MMSE)': [f"{v:+.2f}{'*' if f<0.05 else ''}"
                                     for v,f in zip(l4['GSEA_NES'], l4['GSEA_FDR'])],
     'MMSE corr\n(−rho)':          [f"{-v:+.2f}{'*' if p<0.05 else ''}"
                                     for v,p in zip(l4['MMSE_rho'], l4['MMSE_p'])]},
    index=rows,
)

print(mat.dtypes)   # confirm float64 before plotting

fig, ax = plt.subplots(figsize=(7,4))
sns.heatmap(mat, cmap='RdBu_r', center=0, vmin=-3.2, vmax=3.2,
            annot=ann.values, fmt='', annot_kws={'fontsize':10},
            linewidths=1, linecolor='white',
            cbar_kws={'label':'warm = up in impairment','shrink':0.7}, ax=ax)
ax.set_title('L4 IT CEMiTool modules vs cognition\n(* = FDR/p < 0.05; Path B all non-significant)',
             fontsize=11, pad=12)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0, fontsize=9)
ax.set_ylabel(''); plt.tight_layout(); plt.show()